灵活运用nn.Module实现自定义层块

In [1]:
import torch
from torch import nn
from torch.nn import functional as F

net = nn.Sequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))
X = torch.rand((2, 20))
net(X)

tensor([[-0.0644, -0.4423,  0.0488, -0.0736, -0.0636,  0.1577,  0.0600, -0.0622,
         -0.0120,  0.2826],
        [-0.2060, -0.3293, -0.0178,  0.0585, -0.1503,  0.0644, -0.0103, -0.0103,
         -0.0098,  0.3791]], grad_fn=<AddmmBackward0>)

自定义块

In [2]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(20, 256)
        self.out = nn.Linear(256, 10)

    def forward(self, X):
        return self.out(F.relu(self.linear(X)))
net = MLP()
net(X)


tensor([[-0.3126,  0.1954, -0.0414,  0.0307,  0.1011, -0.0644, -0.1505, -0.0644,
          0.2957,  0.2015],
        [-0.1941,  0.2525, -0.1096, -0.0326, -0.0440, -0.1820, -0.1768, -0.1301,
          0.1858,  0.0774]], grad_fn=<AddmmBackward0>)

顺序块

In [3]:
class MySequential(nn.Module):
    def __init__(self, *args):
        super().__init__()
        for block in args:
            self._modules[block] = block

    def forward(self, X):
        for block in self._modules.values():
            X = block(X)
        return X
net = MySequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))
net(X)

tensor([[-0.1235,  0.0534, -0.1590, -0.1132, -0.0193, -0.0779,  0.2158,  0.0640,
         -0.1411,  0.1759],
        [-0.1312,  0.0018, -0.1247, -0.0891, -0.0509, -0.1011,  0.2293,  0.1576,
         -0.0902,  0.0638]], grad_fn=<AddmmBackward0>)

在正向传播函数中执行代码

In [4]:
class FixedHiddenMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.rand_weight = torch.rand((20, 20), requires_grad = False)
        self.linear = nn.Linear(20, 20)

    def forward(self, X):
        X = self.linear(X)
        X = F.relu(torch.mm(X, self.rand_weight) + 1)
        X = self.linear(X)
        while X.abs().sum() > 2:
            X /= 2
        return X.sum()
net = FixedHiddenMLP()
net(X)

tensor(-0.1386, grad_fn=<SumBackward0>)

混合搭配各种组合块的方法

In [5]:
class NestMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(20, 64), nn.ReLU(), 
                                 nn.Linear(64, 32), nn.ReLU())
        self.linear = nn.Linear(32, 16)

    def forward(self, X):
        return self.linear(self.net(X))
chimera = nn.Sequential(NestMLP(), nn.Linear(16, 20), FixedHiddenMLP())
chimera(X)

tensor(0.0650, grad_fn=<SumBackward0>)